# Distributed Alignment Search (DAS): An End-to-End Tutorial

**CS221M final project — Phase 1 (replication of course material).**

This notebook is a guided, runnable replication of the DAS causal-abstraction
pipeline (Geiger et al., 2023), built from the course notebooks. It has two
working centerpieces, in the order we recommend reading them:

1. **MLP DAS core** — the canonical *hierarchical-equality* task, where DAS
   learns an orthonormal rotation that cleanly partitions an MLP's hidden layer
   into the two intermediate variables `E1` and `E2`. (Source: lecture
   `10_causal_abstraction_ii`.)
2. **LM patching + DAS** — activation patching as a coarse layer x token-position
   baseline, then DAS to localize a factual attribute into a low-dimensional
   subspace of a real language model. (Source: lecture `08_causal_mediation_analysis`.)

**Main metric throughout: Interchange Intervention Accuracy (IIA)** — after we
swap an internal subspace from a *source* run into a *base* run, does the model
produce the counterfactual output predicted by the high-level causal model?

> Run on a **GPU runtime** (Colab: Runtime -> Change runtime type -> GPU).


## Part 0 — Setup

We install `pyvene` (the canonical DAS library from the paper authors; used for
the MLP section) plus the standard HF stack (used for the LM section). The LM
section uses plain forward hooks rather than a heavy framework, so the two parts
stay independent and debuggable.

In [ ]:
# Colab setup. Safe to re-run.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    %pip install -q pyvene==0.1.8 "transformers>=4.44" datasets matplotlib

In [ ]:
import math
import random
import itertools

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else (
    "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE == "cpu":
    print("WARNING: no GPU detected. The MLP section will be slow; reduce N_TRAIN / N_CF below.")

## Part 1 — The idea in one page

**High-level causal model.** A causal model $\mathcal{M}$ is a set of variables
with mechanisms (functions of their parents). Our running example is the
*hierarchical equality* circuit:

- Inputs $A_1, A_2, A_3, A_4 \in \{0,1\}$.
- $E_1 = \mathrm{eq}(A_1, A_2)$, $E_2 = \mathrm{eq}(A_3, A_4)$.
- Output $C = \mathrm{eq}(E_1, E_2)$.

$E_1$ and $E_2$ are the *intermediate variables* we want to find inside a neural
network.

**Interchange intervention.** Take a `base` input and a `source` input. Run both.
Now overwrite the network's representation of (say) $E_1$ with the value it took
on the `source`, and finish the `base` forward pass. If the network truly
implements $\mathcal{M}$, the output should change to exactly what
$\mathcal{M}$ predicts when you set $E_1$ to the source value.

**IIA (Interchange Intervention Accuracy).** The fraction of (base, source) pairs
on which the post-intervention output matches the counterfactual label predicted
by $\mathcal{M}$. IIA = 1.0 means the chosen representation *is* the causal
variable.

**Activation patching** is the coarse version: overwrite the *entire* residual
stream at one (layer, token) location and measure IIA. It localizes information
to a layer/position but not to a *direction*.

**DAS** is the refined version: learn an orthonormal rotation $R$ of the hidden
state and patch only the first $k$ rotated coordinates. The learned subspace can
encode a variable while being orthogonal to others. As we will see in Part 2, $R$
plays the role of a **translation** $\tau$ between the network's coordinates and
the causal model's variables.

## Part 2 — MLP DAS core (centerpiece)

We train a small MLP to solve hierarchical equality, then use DAS to show its
hidden layer factorizes into an `E1` subspace and an `E2` subspace.

### 2.1 The high-level causal model

We implement $\mathcal{M}$ directly in a few lines of Python (mirroring the
`causalab` `CausalModel` used in lecture `10_causal_abstraction_ii`). A *trace* is
just a dict holding every variable's value for one input.

In [ ]:
def eq(a, b):
    "Equality on binary values: 1 if equal else 0."
    return int(a == b)

def sample_trace():
    "Sample one full run of the hierarchical-equality model M."
    a = {f"A{i}": random.randint(0, 1) for i in range(1, 5)}
    E1 = eq(a["A1"], a["A2"])
    E2 = eq(a["A3"], a["A4"])
    C = eq(E1, E2)
    return {**a, "E1": E1, "E2": E2, "C": C}

# sanity check: M on input (1,1,0,0) -> E1=1, E2=1, C=1
t = {"A1": 1, "A2": 1, "A3": 0, "A4": 0}
E1 = eq(t["A1"], t["A2"]); E2 = eq(t["A3"], t["A4"])
print("Input (1,1,0,0):  E1 =", E1, " E2 =", E2, " C =", eq(E1, E2))

### 2.2 Encoding inputs for the MLP

Each example uses **four fresh random vectors**. Positions in the same pair get
the *same* vector when their bits are equal and *different* vectors otherwise;
the two pairs always use distinct vectors. So the MLP can only detect equality by
comparing positions *within* a pair — exactly matching the causal structure of
$\mathcal{M}$. The 4 vectors of dim 4 concatenate into a 16-dim input.

In [ ]:
EMB_DIM = 4  # per-entity embedding dim; MLP input dim = EMB_DIM * 4 = 16

def randvec(n=EMB_DIM, lo=-1, hi=1):
    return np.array([round(random.uniform(lo, hi), 2) for _ in range(n)])

def make_input_tensor(trace):
    "Map an M trace to a 16-dim MLP input with fresh random entity vectors."
    e = [randvec() for _ in range(4)]
    vecs = [None] * 4
    vecs[0] = e[0]
    vecs[1] = e[0] if trace["A1"] == trace["A2"] else e[1]
    vecs[2] = e[2]
    vecs[3] = e[2] if trace["A3"] == trace["A4"] else e[3]
    return torch.tensor(np.concatenate(vecs), dtype=torch.float32)

print("MLP input dim:", EMB_DIM * 4)

### 2.3 Train the MLP

We use pyvene's `create_mlp_classifier` so the trained model plugs directly into
`IntervenableModel` later. Scale `N_TRAIN` down if you are on CPU.

In [ ]:
from pyvene.models.mlp.modelings_mlp import MLPConfig
from pyvene import create_mlp_classifier

H_DIM = 16
N_TRAIN = 262144      # lecture uses 1048576; smaller is plenty here
BATCH = 1024
N_EPOCHS = 3

def generate_factual_data(n):
    X, y = [], []
    for _ in range(n):
        tr = sample_trace()
        X.append(make_input_tensor(tr))
        y.append(tr["C"])
    return torch.stack(X), torch.tensor(y, dtype=torch.long)

X_train, y_train = generate_factual_data(N_TRAIN)
print("Train:", X_train.shape, " positives:", (y_train == 1).sum().item())

mlp_config = MLPConfig(h_dim=H_DIM, n_layer=3, pdrop=0.0, include_bias=True,
                       num_classes=2, squeeze_output=False, input_dim=EMB_DIM * 4)
_, _, mlp = create_mlp_classifier(mlp_config)

# Kaiming init for clean gradient flow through ReLU.
for name, p in mlp.named_parameters():
    if "weight" in name and p.dim() >= 2:
        torch.nn.init.kaiming_normal_(p, nonlinearity="relu")
    elif "bias" in name:
        torch.nn.init.zeros_(p)

mlp.to(DEVICE)
opt = torch.optim.Adam(mlp.parameters(), lr=0.01)

for epoch in range(N_EPOCHS):
    correct = total = 0
    perm = torch.randperm(X_train.shape[0])
    for s in range(0, X_train.shape[0], BATCH):
        idx = perm[s:s + BATCH]
        xb, yb = X_train[idx].to(DEVICE), y_train[idx].to(DEVICE)
        loss, logits = mlp(inputs_embeds=xb, labels=yb)
        opt.zero_grad(); loss.backward(); opt.step()
        correct += (logits.argmax(1) == yb).sum().item(); total += yb.shape[0]
    print(f"epoch {epoch+1}/{N_EPOCHS}  train acc {correct/total:.4f}")

In [ ]:
# Held-out factual accuracy
X_test, y_test = generate_factual_data(10000)
X_test, y_test = X_test.to(DEVICE), y_test.to(DEVICE)
mlp.eval()
with torch.no_grad():
    acc = (mlp(inputs_embeds=X_test)[0].argmax(1) == y_test).float().mean().item()
print(f"Held-out factual accuracy: {acc:.4f}")
print("The MLP solves the task -- but we don't yet know HOW its hidden layer encodes E1, E2.")

### 2.4 Counterfactual data for DAS

For each example we sample a `base` and two `source` inputs and compute the
counterfactual label after interchanging `E1` (from source 1) and/or `E2` (from
source 2). `intervention_id`: `0` = swap E1 only, `1` = swap E2 only, `2` = both.

In [ ]:
def generate_counterfactual_data(n, intervention_id=2):
    data = []
    for _ in range(n):
        base = sample_trace(); s1 = sample_trace(); s2 = sample_trace()
        cf_E1 = s1["E1"] if intervention_id in (0, 2) else base["E1"]
        cf_E2 = s2["E2"] if intervention_id in (1, 2) else base["E2"]
        data.append({
            "input_ids": make_input_tensor(base),
            "source_input_ids": torch.stack([make_input_tensor(s1), make_input_tensor(s2)]),
            "labels": torch.tensor(eq(cf_E1, cf_E2), dtype=torch.long),
            "intervention_id": intervention_id,
        })
    return data

N_CF = 256000
CF_BATCH = 6400
train_cf = generate_counterfactual_data(N_CF, intervention_id=2)
print("Counterfactual train examples:", len(train_cf), " keys:", list(train_cf[0].keys()))

### 2.5 The DAS model

We wrap the MLP in an `IntervenableModel` with **two linked** rotated-subspace
interventions on the hidden layer (`block_output`). Both share one rotation `R`
(`intervention_link_key=0`). In the rotated space we reserve dims `0:8` for `E1`
and `8:16` for `E2`.

In [ ]:
from torch.utils.data import DataLoader
from pyvene import (IntervenableModel, RotatedSpaceIntervention,
                    RepresentationConfig, IntervenableConfig)

def create_das_model(mlp, layer):
    config = IntervenableConfig(
        model_type=type(mlp),
        representations=[
            RepresentationConfig(layer, "block_output", "pos", 1,
                                 subspace_partition=None, intervention_link_key=0),
            RepresentationConfig(layer, "block_output", "pos", 1,
                                 subspace_partition=None, intervention_link_key=0),
        ],
        intervention_types=RotatedSpaceIntervention,
    )
    return IntervenableModel(config, mlp, use_fast=True)

def run_das_forward(das_model, batch, bs, emb=EMB_DIM):
    "Dispatch the interchange by intervention_id. E1 -> dims 0:8, E2 -> dims 8:16."
    iid = batch["intervention_id"][0].item()
    if iid == 2:
        _, cf = das_model(
            {"inputs_embeds": batch["input_ids"]},
            [{"inputs_embeds": batch["source_input_ids"][:, 0]},
             {"inputs_embeds": batch["source_input_ids"][:, 1]}],
            {"sources->base": ([[[0]] * bs, [[0]] * bs], [[[0]] * bs, [[0]] * bs])},
            subspaces=[[[i for i in range(0, emb * 2)]] * bs,
                       [[i for i in range(emb * 2, emb * 4)]] * bs])
    elif iid == 0:
        _, cf = das_model(
            {"inputs_embeds": batch["input_ids"]},
            [{"inputs_embeds": batch["source_input_ids"][:, 0]}, None],
            {"sources->base": ([[[0]] * bs, None], [[[0]] * bs, None])},
            subspaces=[[[i for i in range(0, emb * 2)]] * bs, None])
    elif iid == 1:
        _, cf = das_model(
            {"inputs_embeds": batch["input_ids"]},
            [None, {"inputs_embeds": batch["source_input_ids"][:, 1]}],
            {"sources->base": ([None, [[0]] * bs], [None, [[0]] * bs])},
            subspaces=[None, [[i for i in range(emb * 2, emb * 4)]] * bs])
    return cf

def batched_random_sampler(data, batch_size):
    order = list(range(len(data) // batch_size))
    random.shuffle(order)
    for b in order:
        for i in range(b * batch_size, (b + 1) * batch_size):
            yield i

print("DAS helpers ready.")

### 2.6 Train the rotation R

Only `R` is trainable; the MLP is frozen. We orthonormalize via pyvene's rotated
intervention and clip gradients, exactly as in the lecture.

In [ ]:
das_model = create_das_model(mlp, layer=0)
das_model.set_device(DEVICE)
das_model.disable_model_gradients()

opt_params = []
for k, v in das_model.interventions.items():
    opt_params += [{"params": v.rotate_layer.parameters()}]
    break  # linked: one shared rotation
das_opt = torch.optim.Adam(opt_params, lr=0.001)
print("Trainable params:", das_model.count_parameters())

DAS_EPOCHS = 10
for epoch in range(DAS_EPOCHS):
    das_model.model.train()
    ep_loss = ep_correct = ep_total = 0
    loader = DataLoader(train_cf, batch_size=CF_BATCH,
                        sampler=batched_random_sampler(train_cf, CF_BATCH))
    for batch in loader:
        batch["input_ids"] = batch["input_ids"].unsqueeze(1)
        batch["source_input_ids"] = batch["source_input_ids"].unsqueeze(2)
        bs = batch["input_ids"].shape[0]
        for k, v in batch.items():
            if isinstance(v, torch.Tensor):
                batch[k] = v.to(DEVICE)
        cf = run_das_forward(das_model, batch, bs)
        logits = cf[0].squeeze(1)
        labels = batch["labels"].squeeze().long()
        loss = torch.nn.CrossEntropyLoss()(logits, labels)
        das_opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(das_model.parameters(), 1.0)
        das_opt.step(); das_model.set_zero_grad()
        ep_loss += loss.item()
        ep_correct += (logits.argmax(1) == labels).sum().item(); ep_total += bs
    print(f"epoch {epoch+1}/{DAS_EPOCHS}  loss {ep_loss/(ep_total/CF_BATCH):.4f}  IIA {ep_correct/ep_total:.4f}")

### 2.7 Evaluate IIA (overall and per-variable)

High per-variable IIA confirms the rotation cleanly partitions the hidden layer:
dims `0:8` carry `E1`, dims `8:16` carry `E2`.

In [ ]:
def eval_iia(das_model, intervention_id, n=12800, batch_size=CF_BATCH):
    ds = generate_counterfactual_data(n, intervention_id=intervention_id)
    correct = total = 0
    with torch.no_grad():
        loader = DataLoader(ds, batch_size=batch_size,
                            sampler=batched_random_sampler(ds, batch_size))
        for batch in loader:
            batch["input_ids"] = batch["input_ids"].unsqueeze(1)
            batch["source_input_ids"] = batch["source_input_ids"].unsqueeze(2)
            bs = batch["input_ids"].shape[0]
            for k, v in batch.items():
                if isinstance(v, torch.Tensor):
                    batch[k] = v.to(DEVICE)
            cf = run_das_forward(das_model, batch, bs)
            logits = cf[0].squeeze(1)
            labels = batch["labels"].squeeze().long()
            correct += (logits.argmax(1) == labels).sum().item(); total += bs
    return correct / total

das_model.model.eval()
iia_e1 = eval_iia(das_model, 0)
iia_e2 = eval_iia(das_model, 1)
iia_both = eval_iia(das_model, 2)
print(f"Per-variable IIA:\n  E1 only:      {iia_e1:.4f}\n  E2 only:      {iia_e2:.4f}\n  Both E1 & E2: {iia_both:.4f}")

### 2.8 Ablation A — DAS vs. an untrained (random) rotation

A fresh `IntervenableModel` has a random orthonormal rotation. Its IIA is the
"does any random 8-dim subspace already encode E1/E2?" baseline. The gap to the
trained rotation is what DAS bought us.

In [ ]:
rand_das = create_das_model(mlp, layer=0)
rand_das.set_device(DEVICE)
rand_das.disable_model_gradients()
rand_das.model.eval()
rand_both = eval_iia(rand_das, 2)
rand_e1 = eval_iia(rand_das, 0)
rand_e2 = eval_iia(rand_das, 1)
print(f"Random rotation IIA -> E1: {rand_e1:.4f}  E2: {rand_e2:.4f}  both: {rand_both:.4f}")
print(f"Trained rotation IIA -> E1: {iia_e1:.4f}  E2: {iia_e2:.4f}  both: {iia_both:.4f}")

fig, ax = plt.subplots(figsize=(6, 4))
labels = ["E1 only", "E2 only", "Both"]
x = np.arange(len(labels)); w = 0.35
ax.bar(x - w/2, [rand_e1, rand_e2, rand_both], w, label="Random rotation")
ax.bar(x + w/2, [iia_e1, iia_e2, iia_both], w, label="Trained DAS")
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylabel("IIA"); ax.set_ylim(0, 1.05)
ax.axhline(0.5, ls="--", c="gray", alpha=0.6, label="chance")
ax.set_title("DAS localizes E1/E2; a random subspace does not")
ax.legend(); plt.tight_layout(); plt.show()

### 2.9 Ablation B — subspace-dimension sweep

How many rotated dimensions does `E1` actually need? We retrain DAS with the
`both` intervention but vary the size `d` of each variable's subspace (so the two
subspaces are dims `0:d` and `d:2d`). This is a quick, poster-friendly curve;
keep `SWEEP_EPOCHS` small.

In [ ]:
def create_das_model_partition(mlp, layer, d):
    "Two linked rotated interventions; each variable owns d dims."
    config = IntervenableConfig(
        model_type=type(mlp),
        representations=[
            RepresentationConfig(layer, "block_output", "pos", 1,
                                 subspace_partition=None, intervention_link_key=0),
            RepresentationConfig(layer, "block_output", "pos", 1,
                                 subspace_partition=None, intervention_link_key=0),
        ],
        intervention_types=RotatedSpaceIntervention,
    )
    return IntervenableModel(config, mlp, use_fast=True)

def run_forward_dims(dm, batch, bs, d):
    _, cf = dm(
        {"inputs_embeds": batch["input_ids"]},
        [{"inputs_embeds": batch["source_input_ids"][:, 0]},
         {"inputs_embeds": batch["source_input_ids"][:, 1]}],
        {"sources->base": ([[[0]] * bs, [[0]] * bs], [[[0]] * bs, [[0]] * bs])},
        subspaces=[[[i for i in range(0, d)]] * bs,
                   [[i for i in range(d, 2 * d)]] * bs])
    return cf

def train_eval_dim(d, epochs, n_cf=64000, n_eval=12800):
    dm = create_das_model_partition(mlp, 0, d)
    dm.set_device(DEVICE); dm.disable_model_gradients()
    params = []
    for k, v in dm.interventions.items():
        params += [{"params": v.rotate_layer.parameters()}]; break
    opt = torch.optim.Adam(params, lr=0.001)
    tr = generate_counterfactual_data(n_cf, 2)
    for _ in range(epochs):
        dm.model.train()
        loader = DataLoader(tr, batch_size=CF_BATCH,
                            sampler=batched_random_sampler(tr, CF_BATCH))
        for batch in loader:
            batch["input_ids"] = batch["input_ids"].unsqueeze(1)
            batch["source_input_ids"] = batch["source_input_ids"].unsqueeze(2)
            bs = batch["input_ids"].shape[0]
            for k, v in batch.items():
                if isinstance(v, torch.Tensor):
                    batch[k] = v.to(DEVICE)
            cf = run_forward_dims(dm, batch, bs, d)
            logits = cf[0].squeeze(1); labels = batch["labels"].squeeze().long()
            loss = torch.nn.CrossEntropyLoss()(logits, labels)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(dm.parameters(), 1.0)
            opt.step(); dm.set_zero_grad()
    # eval
    dm.model.eval(); ev = generate_counterfactual_data(n_eval, 2)
    correct = total = 0
    with torch.no_grad():
        loader = DataLoader(ev, batch_size=CF_BATCH,
                            sampler=batched_random_sampler(ev, CF_BATCH))
        for batch in loader:
            batch["input_ids"] = batch["input_ids"].unsqueeze(1)
            batch["source_input_ids"] = batch["source_input_ids"].unsqueeze(2)
            bs = batch["input_ids"].shape[0]
            for k, v in batch.items():
                if isinstance(v, torch.Tensor):
                    batch[k] = v.to(DEVICE)
            cf = run_forward_dims(dm, batch, bs, d)
            logits = cf[0].squeeze(1); labels = batch["labels"].squeeze().long()
            correct += (logits.argmax(1) == labels).sum().item(); total += bs
    return correct / total

SWEEP_EPOCHS = 6
dims = [1, 2, 4, 8]
sweep = [(d, train_eval_dim(d, SWEEP_EPOCHS)) for d in dims]
for d, a in sweep:
    print(f"subspace dim per variable = {d:2d}  ->  IIA {a:.4f}")

ds_, accs_ = zip(*sweep)
plt.figure(figsize=(6, 4))
plt.plot(ds_, accs_, "o-")
plt.xlabel("Subspace dim per variable"); plt.ylabel("IIA (both)")
plt.title("How many dimensions does each equality variable need?")
plt.ylim(0, 1.05); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

### 2.10 Connecting back: R is a learned translation

The rotation $R$ DAS learned plays the role of the translation $\tau$ between the
MLP's coordinates and the causal model's variables. The first 8 rotated
coordinates *are* the network's representation of $E_1$; the next 8 are $E_2$.
High IIA means the constructive-abstraction equation holds: intervening on the
subspace is equivalent to intervening on the high-level variable.

## Part 3 — From toy MLP to a real language model

Now we repeat the logic on a pretrained LM. The high-level variable is the
**country attribute** of a city. Base prompt: *"Paris is in the country of"* ->
`France`. Source prompt: *"Tokyo is in the country of"* -> `Japan`. If we patch
the right place, the base output should flip to `Japan`.

We use plain forward hooks (no extra framework) so the intervention is fully
transparent. Model: `HuggingFaceTB/SmolLM2-360M` (small, knows these facts).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

LM_NAME = "HuggingFaceTB/SmolLM2-360M"
tok = AutoTokenizer.from_pretrained(LM_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
lm = AutoModelForCausalLM.from_pretrained(LM_NAME, torch_dtype=torch.float32).eval().to(DEVICE)

def get_layers(model):
    for attr in ["model.layers", "transformer.h", "model.decoder.layers"]:
        obj = model
        try:
            for p in attr.split("."):
                obj = getattr(obj, p)
            return list(obj)
        except AttributeError:
            continue
    raise ValueError("cannot find layers")

def hidden_size(model):
    for a in ["hidden_size", "n_embd", "d_model"]:
        if hasattr(model.config, a):
            return int(getattr(model.config, a))
    raise ValueError("no hidden size")

N_LAYERS = len(get_layers(lm))
HID = hidden_size(lm)
print(f"{LM_NAME}: {N_LAYERS} layers, hidden {HID}")

def value_token_id(value):
    return tok.encode(" " + value.strip(), add_special_tokens=False)[0]

### 3.1 Build a clean, model-correct dataset

We keep only cities the model already answers correctly, and pair each city with
a source city of a *different* country. We also fix all prompts to the same token
length so a single token position aligns across examples (we use cities that are
single tokens for clean patching).

In [ ]:
CITIES = [
    ("Paris", "France"), ("Tokyo", "Japan"), ("Berlin", "Germany"),
    ("Cairo", "Egypt"), ("Madrid", "Spain"), ("Rome", "Italy"),
    ("Moscow", "Russia"), ("London", "England"), ("Athens", "Greece"),
    ("Dublin", "Ireland"), ("Lisbon", "Portugal"), ("Vienna", "Austria"),
    ("Oslo", "Norway"), ("Warsaw", "Poland"), ("Bangkok", "Thailand"),
]
PROMPT = "{city} is in the country of"

def next_token_id(prompt):
    ids = tok(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        return lm(**ids).logits[0, -1].argmax().item()

# keep cities the model gets right AND whose prompt tokenizes to a fixed length
correct = []
lengths = set()
for city, country in CITIES:
    p = PROMPT.format(city=city)
    L = len(tok(p, add_special_tokens=False)["input_ids"])
    if next_token_id(p) == value_token_id(country):
        correct.append((city, country, L))
        lengths.add(L)
print("Model-correct cities:", [(c, k) for c, k, _ in correct])
# pick the most common prompt length so positions align
from collections import Counter
target_len = Counter([L for *_, L in correct]).most_common(1)[0][0]
correct = [(c, k) for c, k, L in correct if L == target_len]
N_POS = target_len
print(f"Using {len(correct)} cities at prompt length {N_POS} tokens.")

In [ ]:
# Build base/source pairs. Cause label = source country; isolate label = base country.
def build_pairs(items):
    pairs = []
    for i, (bc, bk) in enumerate(items):
        sc, sk = items[(i + 1) % len(items)]  # different city/country
        pairs.append({
            "base_prompt": PROMPT.format(city=bc),
            "source_prompt": PROMPT.format(city=sc),
            "cause_label": value_token_id(sk),    # output should become source country
            "isolate_label": value_token_id(bk),  # output should stay base country
        })
    return pairs

pairs = build_pairs(correct)
print("Example pair:", {k: pairs[0][k] for k in ["base_prompt", "source_prompt"]})

### 3.2 Forward hooks: collect activations and patch

`collect_layer_outputs` caches every layer's residual stream for a prompt.
`forward_with_patch` re-runs the base prompt while overwriting the hidden state at
one `(layer, pos)` with a (possibly transformed) source vector.

In [ ]:
def collect_layer_outputs(prompt):
    layers = get_layers(lm)
    out = [None] * len(layers)
    handles = []
    def mk(li):
        def hook(m, inp, o):
            hs = o[0] if isinstance(o, tuple) else o
            out[li] = hs[0].detach()
        return hook
    for li, layer in enumerate(layers):
        handles.append(layer.register_forward_hook(mk(li)))
    ids = tok(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        lm(**ids)
    for h in handles:
        h.remove()
    return out  # list[layer] -> [seq, hidden]

def forward_with_patch(base_prompt, source_h, layer_idx, pos, transform_fn=None, requires_grad=False):
    layers = get_layers(lm)
    def hook(m, inp, o):
        hs = o[0] if isinstance(o, tuple) else o
        h_base = hs[0, pos, :]
        h_new = transform_fn(h_base, source_h) if transform_fn is not None else source_h
        hs2 = hs.clone(); hs2[0, pos, :] = h_new
        return (hs2,) + o[1:] if isinstance(o, tuple) else hs2
    handle = layers[layer_idx].register_forward_hook(hook)
    ids = tok(base_prompt, return_tensors="pt").to(DEVICE)
    ctx = torch.enable_grad() if requires_grad else torch.no_grad()
    with ctx:
        out = lm(**ids)
    handle.remove()
    return out.logits[0, -1]

# Pre-cache source activations for every pair.
for ex in pairs:
    ex["source_cache"] = collect_layer_outputs(ex["source_prompt"])  # list[layer] [seq,hidden]
print("Cached source activations for", len(pairs), "pairs.")

### 3.3 Activation-patching baseline: layer x position IIA heatmap

For every `(layer, pos)` we patch the full residual vector and measure **cause
IIA** — did the output flip to the *source* country? The bright region tells us
*where* the country attribute lives.

In [ ]:
cause_grid = np.zeros((N_LAYERS, N_POS))
for li in range(N_LAYERS):
    for pos in range(N_POS):
        c = 0
        for ex in pairs:
            src_h = ex["source_cache"][li][pos].to(DEVICE)
            logits = forward_with_patch(ex["base_prompt"], src_h, li, pos)
            if logits.argmax().item() == ex["cause_label"]:
                c += 1
        cause_grid[li, pos] = c / len(pairs)
    print(f"layer {li+1}/{N_LAYERS} done", end="\r")
print()

best_layer, best_pos = np.unravel_index(cause_grid.argmax(), cause_grid.shape)
print(f"Best full-patch cell: layer {best_layer}, pos {best_pos}, cause IIA {cause_grid[best_layer,best_pos]:.3f}")

tok_labels = tok.convert_ids_to_tokens(tok(pairs[0]["base_prompt"], add_special_tokens=False)["input_ids"])
plt.figure(figsize=(max(6, N_POS), max(5, N_LAYERS * 0.3)))
plt.imshow(cause_grid, aspect="auto", origin="lower", cmap="viridis", vmin=0, vmax=1)
plt.colorbar(label="Cause IIA")
plt.xticks(range(N_POS), tok_labels, rotation=45, ha="right")
plt.yticks(range(N_LAYERS))
plt.xlabel("Token position"); plt.ylabel("Layer")
plt.title("Activation patching: where does 'country' live?")
plt.tight_layout(); plt.show()

### 3.4 DAS on the LM: localize the attribute to a subspace

At the best `(layer, pos)` we learn an orthonormal rotation `R` and patch only
the first `k` rotated dimensions. We keep `R` orthonormal by QR-projecting after
each step (the same recipe as the canonical DAS rotation). We train to flip the
output to the source country (cause), then report cause IIA and **isolate IIA**
(did *other* runs keep their base answer? a subspace that is too broad will hurt
this).

In [ ]:
K = 32                       # subspace dim to patch
DAS_LAYER, DAS_POS = int(best_layer), int(best_pos)
LM_DAS_EPOCHS = 12

def project_orthonormal(R):
    Q, _ = torch.linalg.qr(R)
    return Q

R = torch.eye(HID, device=DEVICE, requires_grad=True)
r_opt = torch.optim.Adam([R], lr=1e-3)

for epoch in range(LM_DAS_EPOCHS):
    random.shuffle(pairs)
    total_loss = 0.0
    for ex in pairs:
        src_h = ex["source_cache"][DAS_LAYER][DAS_POS].to(DEVICE)
        R_orth = project_orthonormal(R)
        def transform_fn(h_base, h_src=src_h, Ro=R_orth):
            fb = h_base @ Ro; fs = h_src @ Ro
            fp = torch.cat([fs[:K], fb[K:]], dim=0)
            return fp @ Ro.T
        logits = forward_with_patch(ex["base_prompt"], src_h, DAS_LAYER, DAS_POS,
                                    transform_fn=transform_fn, requires_grad=True)
        loss = F.cross_entropy(logits.unsqueeze(0),
                               torch.tensor([ex["cause_label"]], device=DEVICE))
        r_opt.zero_grad(); loss.backward(); r_opt.step()
        with torch.no_grad():
            R.data = project_orthonormal(R)
        total_loss += loss.item()
    print(f"epoch {epoch+1}/{LM_DAS_EPOCHS}  loss {total_loss/len(pairs):.4f}")

In [ ]:
# Evaluate the learned subspace: cause IIA (flip to source) and isolate IIA (keep base).
def das_eval(R, label_key, k=K):
    R = project_orthonormal(R).detach()
    correct = 0
    for ex in pairs:
        src_h = ex["source_cache"][DAS_LAYER][DAS_POS].to(DEVICE)
        def transform_fn(h_base, h_src=src_h, Ro=R):
            fb = h_base @ Ro; fs = h_src @ Ro
            fp = torch.cat([fs[:k], fb[k:]], dim=0)
            return fp @ Ro.T
        logits = forward_with_patch(ex["base_prompt"], src_h, DAS_LAYER, DAS_POS,
                                    transform_fn=transform_fn)
        if logits.argmax().item() == ex[label_key]:
            correct += 1
    return correct / len(pairs)

das_cause = das_eval(R, "cause_label")
das_isolate = das_eval(R, "isolate_label")
full_cause = cause_grid[DAS_LAYER, DAS_POS]
print(f"At layer {DAS_LAYER}, pos {DAS_POS}:")
print(f"  Full-vector patch   cause IIA: {full_cause:.3f}")
print(f"  DAS ({K}-dim subspace) cause IIA: {das_cause:.3f}   isolate IIA: {das_isolate:.3f}")
print("DAS reaches comparable cause IIA while touching only a {}-dim subspace.".format(K))

### 3.5 What we showed

- **Activation patching** localized the country attribute to a band of
  layers/positions (coarse: a whole residual vector).
- **DAS** found a low-dimensional *direction* at the best location that carries
  the attribute, mirroring the MLP result: the same method scales from a toy MLP
  to a real LM.

The `K`-dim subspace and the layer/position are natural knobs for a poster:
sweep `K`, or compare the patching heatmap against a DAS-per-cell heatmap.

## Part 4 — Where this goes next (Phase 2)

This notebook replicated the course material end-to-end: the MLP hierarchical-
equality DAS and the LM patching+DAS localization. The natural extensions, which
fall outside the core course notebooks, are:

1. **Entity binding with DAS** — find the subspace that binds an entity to its
   attribute in an in-context list ("Pete has jam. Ann has pie. ... Ann has ___"),
   distinguishing the bound value from positional confounds. (Source notebook
   `09_counterfactual_design`.)
2. **Concept DAS** (Bao et al., 2026) — extending the learned-subspace idea to
   higher-level concepts.

Those are Phase 2; we will build them on top of the machinery established here.